# Get sparseGRM files (Python kernel)

In [ ]:
!gsutil -m cp -r gs://fc-secure-f2ecfa45-d7db-4232-929f-8cf417be1eb0/cromwell-execution/FastSparseGRM/***.sGRM.RData ./Aim_3/Rare_variant_PGS/STAAR_pipeline/

# R Setup

In [ ]:
# Setup #
#install.packages("pacman")
library(pacman)

p_load(rlang, RISCA, gdsfmt, SeqArray,
       SeqVarTools, dplyr,
       Matrix, glmnet,
       caret, GenomicFeatures, 
       TxDb.Hsapiens.UCSC.hg38.knownGene,
       BiocManager, tidyverse, data.table,
       R.utils)

#BiocManager::install("GENESIS")
library(GENESIS)

p_load_gh("xihaoli/STAAR",
          "xihaoli/MultiSTAAR",
          "zilinli1988/SCANG",
          "xihaoli/STAARpipeline")

p_load_gh('jwilliams10/RICE')

# Load phenotype data

In [ ]:
pheno = data.table::fread("./Aim_3/phenotypes/aim_3_phenotypes.txt")


breast_training = data.table::fread("./Aim_3/phenotypes/Aim_3_set_assignments/internal_breast_cancer_Training_single.txt")
breast_pheno = pheno %>%
  dplyr::filter(breast_cancer > -1, person_id %in% breast_training$V1) %>%
  dplyr::select(c(person_id,age_at_sample,paste0("PC",1:30),
                  Superpopulation,breast_cancer))

colorectal_training = data.table::fread("./Aim_3/phenotypes/Aim_3_set_assignments/internal_colorectal_cancer_Training_single.txt")
colorectal_pheno = pheno %>%
  dplyr::filter(colorectal_cancer > -1, person_id %in% colorectal_training$V1) %>%
  dplyr::select(c(person_id,Female,age_at_sample,paste0("PC",1:30),
                  Superpopulation,colorectal_cancer))

prostate_training = data.table::fread("./Aim_3/phenotypes/Aim_3_set_assignments/internal_prostate_cancer_Training_single.txt")
prostate_pheno = pheno %>%
  dplyr::filter(prostate_cancer > -1, person_id %in% prostate_training$V1) %>%
  dplyr::select(c(person_id,age_at_sample,paste0("PC",1:30),
                  Superpopulation,prostate_cancer))

rm(pheno)

# Load and merge CV PGS data

In [ ]:
breast_score = fread("/home/jupyter/workspaces/v8geneticepidemiologyofcommoncancersinuspopulations/Aim_3/Common_variant_PGS/scores/breast_scores.sscore") %>%
  dplyr::select(IID,SCORE1_AVG)
colnames(breast_score) = c("person_id","breast_PGS")
colorectal_score = fread("/home/jupyter/workspaces/v8geneticepidemiologyofcommoncancersinuspopulations/Aim_3/Common_variant_PGS/scores/colorectal_scores.sscore")%>%
  dplyr::select(IID,SCORE1_AVG)
colnames(colorectal_score) = c("person_id","colorectal_PGS")
prostate_score = fread("/home/jupyter/workspaces/v8geneticepidemiologyofcommoncancersinuspopulations/Aim_3/Common_variant_PGS/scores/prostate_scores.sscore")%>%
  dplyr::select(IID,SCORE1_AVG)
colnames(prostate_score) = c("person_id","prostate_PGS")

breast_pheno = merge(breast_pheno,breast_score)
colorectal_pheno = merge(colorectal_pheno,colorectal_score)
prostate_pheno = merge(prostate_pheno,prostate_score)

# Run PCA-based PGS normalization function

## Define function

In [ ]:
# Function #
normalize_pgs_pca <- function(data, pgs_col, pca_cols, ref_filter = NULL) {
  # --- 1. Define reference population ---
  if (!is.null(ref_filter)) {
    ref_data <- data %>% filter(!!parse_expr(ref_filter))
  } else {
    ref_data <- data
  }
  
  # --- 2. Fit regression model for mean adjustment (Z_norm1) ---
  formula_mean <- as.formula(paste(pgs_col, "~", paste(pca_cols, collapse = " + ")))
  mean_model <- lm(formula_mean, data = ref_data)
  
  # Predicted mean PGS for all individuals
  data$pred_mean <- predict(mean_model, newdata = data)
  
  # Residuals
  data$resid_mean <- data[[pgs_col]] - data$pred_mean
  
  # SD of residuals in reference population
  if (!is.null(ref_filter)) {
    sd_ref <- sd(data$resid_mean[with(data, eval(parse(text = ref_filter)))], na.rm = TRUE)
  } else {
    sd_ref <- sd(data$resid_mean, na.rm = TRUE)
  }
  
  # Z_norm1: mean-adjusted
  data$Z_norm1 <- data$resid_mean / sd_ref
  
  # --- 3. Fit regression model for variance adjustment (Z_norm2) ---
  # Create ref_data with residuals for variance model
  ref_data <- if (!is.null(ref_filter)) {
    data %>% filter(!!parse_expr(ref_filter))
  } else {
    data
  }
  
  ref_data <- ref_data %>% mutate(resid_sq = resid_mean^2)
  formula_sd <- as.formula(paste("resid_sq ~", paste(pca_cols, collapse = " + ")))
  sd_model <- lm(formula_sd, data = ref_data)
  
  # Predicted SD for all individuals
  data$pred_var <- predict(sd_model, newdata = data)
  data$pred_sd <- sqrt(pmax(data$pred_var, 1e-8))
  
  # Z_norm2: mean + variance normalized
  data$Z_norm2 <- data$resid_mean / data$pred_sd
  
  return(data)
}


## Run function

In [ ]:
breast_pheno <- normalize_pgs_pca(
  breast_pheno,
  pgs_col = "breast_PGS",
  pca_cols = paste0("PC", 1:30),
  ref_filter = "breast_cancer == 0"
)

colorectal_pheno <- normalize_pgs_pca(
  colorectal_pheno,
  pgs_col = "colorectal_PGS",
  pca_cols = paste0("PC", 1:30),
  ref_filter = "colorectal_cancer == 0"
)

prostate_pheno <- normalize_pgs_pca(
  prostate_pheno,
  pgs_col = "prostate_PGS",
  pca_cols = paste0("PC", 1:30),
  ref_filter = "prostate_cancer == 0"
)


## Check resulting PGS

In [ ]:
ggplot(data = colorectal_pheno, aes(x = Z_norm2, fill = Superpopulation)) + geom_density(alpha = 0.4)

# Prepare phenotype datasets for null models

In [ ]:
## Breast ##

STAARpipeline_breast_training_data = breast_pheno %>%
  dplyr::select(c(person_id,breast_cancer,age_at_sample,Z_norm2,paste0("PC",1:30)))
colnames(STAARpipeline_breast_training_data) = c("person_id","breast_cancer","Age","Standardized_CV_PRS",paste0("PC",1:30))

write.csv(STAARpipeline_breast_training_data,"./Aim_3/Rare_variant_PGS/Phenotypes/STAAR_pipeline_breast_cancer_phenotypes.csv", row.names = FALSE, quote = FALSE)


## Colorectal ##
STAARpipeline_colorectal_training_data = colorectal_pheno %>%
  dplyr::select(c(person_id,colorectal_cancer,age_at_sample,Female,Z_norm2,paste0("PC",1:30)))

colnames(STAARpipeline_colorectal_training_data) = c("person_id","colorectal_cancer","Age","Female","Standardized_CV_PRS",paste0("PC",1:30))

write.csv(STAARpipeline_colorectal_training_data,"./Aim_3/Rare_variant_PGS/Phenotypes/STAAR_pipeline_colorectal_cancer_phenotypes.csv", row.names = FALSE, quote = FALSE)


## Prostate ##
STAARpipeline_prostate_training_data = prostate_pheno %>%
  dplyr::select(c(person_id,prostate_cancer,age_at_sample,Z_norm2,paste0("PC",1:30)))

colnames(STAARpipeline_prostate_training_data) = c("person_id","prostate_cancer","Age","Standardized_CV_PRS",paste0("PC",1:30))

write.csv(STAARpipeline_prostate_training_data,"./Aim_3/Rare_variant_PGS/Phenotypes/STAAR_pipeline_prostate_cancer_phenotypes.csv", row.names = FALSE, quote = FALSE)


# Fit null models

## Breast

In [ ]:
## sparseGRM ##
load("./Aim_3/Rare_variant_PGS/STAAR_pipeline/breast_SparseGRM.sGRM.RData")

## Phenotypes ##
STAARpipeline_breast_training_data = STAARpipeline_breast_training_data %>%
  mutate(IDs = paste0(0, "_", person_id))

# Fit null model #
vars <- c("Age", "Standardized_CV_PRS", paste0("PC", 1:30))
formula_obj <- reformulate(vars, response = "breast_cancer")

Null_Model <- fit_nullmodel(fixed = formula_obj,
                            data = STAARpipeline_breast_training_data,
                            id = "IDs",
                            kins = sGRM,
                            family = binomial(link = "logit"),
                            use_SPA = TRUE,
                            verbose = TRUE)

In [ ]:
rm(sGRM)

In [ ]:
saveRDS(Null_Model, "./Aim_3/Rare_variant_PGS/STAAR_pipeline/STAAR_pipeline_breast_cancer_null_model.rds")

## Colorectal

In [ ]:
## sparseGRM ##
load("./Aim_3/Rare_variant_PGS/STAAR_pipeline/colorectal_SparseGRM.sGRM.RData")

## Phenotypes ##
STAARpipeline_colorectal_training_data = STAARpipeline_colorectal_training_data %>%
  mutate(IDs = paste0(0, "_", person_id))

# Fit null model #
vars <- c("Age","Female", "Standardized_CV_PRS", paste0("PC", 1:30))
formula_obj <- reformulate(vars, response = "colorectal_cancer")

Null_Model <- fit_nullmodel(fixed = formula_obj,
                            data = STAARpipeline_colorectal_training_data,
                            id = "IDs",
                            kins = sGRM,
                            family = binomial(link = "logit"),
                            use_SPA = TRUE,
                            verbose = TRUE)

In [ ]:
rm(sGRM)

In [ ]:
saveRDS(Null_Model, "./Aim_3/Rare_variant_PGS/STAAR_pipeline/STAAR_pipeline_colorectal_cancer_null_model.rds")

## Prostate

In [ ]:
## sparseGRM ##
load("./Aim_3/Rare_variant_PGS/STAAR_pipeline/prostate_SparseGRM.sGRM.RData")

## Phenotypes ##
STAARpipeline_prostate_training_data = STAARpipeline_prostate_training_data %>%
  mutate(IDs = paste0(0, "_", person_id))

# Fit null model #
vars <- c("Age", "Standardized_CV_PRS", paste0("PC", 1:30))
formula_obj <- reformulate(vars, response = "prostate_cancer")

Null_Model <- fit_nullmodel(fixed = formula_obj,
                            data = STAARpipeline_prostate_training_data,
                            id = "IDs",
                            kins = sGRM,
                            family = binomial(link = "logit"),
                            use_SPA = TRUE,
                            verbose = TRUE)

In [ ]:
rm(sGRM)

In [ ]:
saveRDS(Null_Model, "./Aim_3/Rare_variant_PGS/STAAR_pipeline/STAAR_pipeline_prostate_cancer_null_model.rds")

# Upload null models to bucket (Python kernel)

In [ ]:
!gsutil -m cp -r ./Aim_3/Rare_variant_PGS/STAAR_pipeline/***null_model.rds gs://fc-secure-f2ecfa45-d7db-4232-929f-8cf417be1eb0/austin_working/dissertation_project/Aim_3/Rare_variant_PGS/STAAR_pipeline/